# 초기 검색 방식 비교

KoE5/Qwen 기반의 질문 재작성·HyDE·재순위화 탐색. 초기 실험입니다.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


# 구글 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 라이브러리 설치

In [ ]:
!pip install -qU langchain-text-splitters langchain-community langgraph langchain-huggingface faiss-cpu bitsandbytes accelerate pypdf
!pip install pdfplumber

# 문서 파일 불러오기

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/drive/MyDrive/2024년+원천징수의무자를+위한+연말정산+신고안내 (1).pdf"
loader = PyPDFLoader(file_path)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

# 문서 내용 확인

In [ ]:
for i in range(5):
  print(f"번호: {i}")
  print(pages[i].page_content[:])
  print()

In [ ]:
pages[0].metadata

In [ ]:
import re

def preprocess_text(text):
    # 앞뒤 공백 제거
    text = text.strip()

    # 스페이스/탭이 여러 개 연속되면 하나로
    text = re.sub(r'[ \t]+', ' ', text)

    # 줄바꿈이 3개 이상이면 2개로
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text

In [ ]:
prepared_pages = []
for page in pages:
  prepared_pages.append(preprocess_text(page.page_content))

In [ ]:
for i in range(5):
  print(f"번호: {i}")
  print(prepared_pages[i])
  print()

# 문서 청킹

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

all_splits = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

for page_num, prepared_page in enumerate(prepared_pages):
    chunks = text_splitter.split_text(prepared_page)

    for chunk in chunks:
        all_splits.append(
            Document(
                page_content=chunk,
                metadata={"page": page_num}
            )
        )

In [ ]:
for i in range(5):
  print(f"번호: {i+1}")
  print(all_splits[i])
  print()

In [ ]:
# PDF 11·12쪽 제외: metadata의 page 값은 각각 10·11
search_splits = [
    doc for doc in all_splits
    if doc.metadata["page"] not in {10, 11, 12, 13}
]

# Vector DB에 저장

In [ ]:
import faiss
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# 임베딩 모델 이름
embedding_model_name = "nlpai-lab/KoE5"
# 임베딩 모델
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
)

# 임베딩 된 값 저장소
embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings, # 임베딩 모델 설정
    index=index, # 임베딩 된 값 저장소 설정
    docstore=InMemoryDocstore(), # 원본 문서 저장소 설정
    index_to_docstore_id={}, # 벡터와 저장소 대응
)
# 임베딩 되어서 저장됨
ids = vector_store.add_documents(documents=search_splits)

# 질문 임베딩 객체
retriever = vector_store.as_retriever()

# 모델 및 토크나이저 세팅

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

language_model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=False,
)
model = AutoModelForCausalLM.from_pretrained(
    language_model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name)

In [ ]:
from transformers import pipeline

llm_pipeline = pipeline(
    model=model, # 모델 설정
    tokenizer=tokenizer, # 토크나이저 설정
    task="text-generation", # 글 생성
    do_sample=False, # 확률이 높은 것을 뽑아 사용 (같은 질문의 결과를 비교하기 위해 사용)
    repetition_penalty=1.2, # 어느정도 같은 문구 반복 허용
    return_full_text=False, # 답변만 반환
    max_new_tokens=220, # 생성 토큰 수 제한 (좀 더 높여도 괜찮을 것 같다.)
)

# 파이프라인을 기준으로 llm 설정
llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)

# RAG 검색 방식과 재순위화 비교

같은 문서, 임베딩, 생성 모델, 답변 프롬프트를 사용합니다. 기본 검색·Query Rewriting·HyDE를 각각 실행한 뒤, **같은 후보 8개**에서 재순위화 전후 상위 4개와 답변을 비교합니다.

재순위화는 질문과 후보 문서를 함께 채점하는 Cross-Encoder 한 단계만 추가합니다. 필터링, 압축, 중복 제거는 적용하지 않습니다.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 세 방법 모두 이 답변 프롬프트와 같은 chat_model을 사용합니다.
answer_template = """너는 제공된 문서를 근거로 질문에 답하는 도우미야.

아래 맥락에서 질문과 직접 관련된 내용만 사용해 답해.

- 문서에 없는 사실이나 추측을 추가하지 마.
- 문서에 적힌 대상, 조건, 예외, 시점을 구분해. 특히 허용과 금지처럼 의미가 반대인 표현을 바꾸어 설명하지 마.
- 질문의 일부만 확인된다면 확인되는 부분만 답하고, 확인되지 않는 부분은 확인할 수 없다고 밝혀.
- 관련 근거가 전혀 없으면 "제공된 문서에서 확인할 수 없습니다."라고만 답해.
- 반복이나 일반적인 조언 없이, 필요한 만큼만 한국어로 간결하게 답해.

맥락:
{context}

질문:
{question}

답변:
"""
answer_prompt = PromptTemplate.from_template(answer_template)
answer_chain = answer_prompt | chat_model | StrOutputParser()


## 1. 기본 검색

원래 질문을 그대로 검색기에 전달합니다. 검색어 생성을 위한 모델 호출이 없습니다.


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def answer_from_search(question, search_query):
    """세 방법에 공통으로 쓰는 검색 및 답변 생성 단계."""
    # 문서 찾기
    docs = retriever.invoke(search_query)
    # 문서를 context에 질문을 question에 넣고 템플릿 만들기 -> 답변 부분 생성
    answer = answer_chain.invoke({
        "context": format_docs(docs),
        "question": question,  # 답변할 때는 항상 원래 질문을 사용
    })
    return {
        "question": question,
        "search_query": search_query,
        "docs": docs,
        "answer": answer,
    }


def run_baseline(question):
  # 같은 질문을 문서 찾기와 답변하기에 모두 넣음
    return answer_from_search(question, question)


## 2. Query Rewriting

LLM이 원래 질문을 한 줄 검색어로 바꿉니다. 바뀐 검색어로 검색하되, 답변 생성에는 원래 질문을 사용합니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 연말정산 PDF의 검색어 변환기입니다.
질문에 답하거나 세법 내용을 추론하지 마세요. 질문에 이미 들어 있는 핵심 개념만 공식 용어로 바꾸세요.
질문에 없는 사람, 공제, 신고 의무, 금액, 날짜, 사례를 추가하지 마세요.
출력은 검색용 명사구 한 줄만 작성하세요. 완전한 문장, 설명, 목록, 따옴표는 금지합니다.

형식 예시:
질문: 카드 엄청 썼는데 공제돼?
출력: 신용카드 사용금액 소득공제 요건
질문: 부모님도 공제 가능해?
출력: 직계존속 기본공제 요건"""),
    ("human", "질문: {question}\n출력:"),
])
rewrite_query_chain = rewrite_prompt | chat_model | StrOutputParser()


def run_rewrite(question):
    search_query = rewrite_query_chain.invoke({"question": question}).strip()
    return answer_from_search(question, search_query)


## 3. HyDE

LLM이 검색에 사용할 짧은 가상 문서를 작성합니다. 이는 실제 근거 문서가 아니며, **검색 입력에만** 사용합니다. 최종 답변의 근거는 Retriever가 가져온 실제 문서입니다.


In [ ]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 연말정산 PDF에서 관련 문단을 찾기 위한 가상 문서 작성자입니다.
실제 답변을 작성하는 단계가 아닙니다. 질문의 핵심 주제어를 그대로 포함한 안내문 1문장만 작성하세요.
질문에 없는 다른 공제 항목이나 부동산·월세 등 다른 주제로 이동하지 마세요.
확인되지 않은 법률 조문, 수치, 기간, 신고 의무, 자격 요건을 단정하지 마세요.
한 문장 이외의 설명, 목록, 제목, 따옴표는 금지합니다.

형식 예시:
질문: 기부금 공제 때 주의할 점은?
출력: 연말정산에서 기부금 공제의 적용 범위와 증빙서류 확인 사항을 안내한다."""),
    ("human", "질문: {question}\n출력:"),
])
hyde_query_chain = hyde_prompt | chat_model | StrOutputParser()


def run_hyde(question):
    search_query = hyde_query_chain.invoke({"question": question}).strip()
    return answer_from_search(question, search_query)


## 4. Cross-Encoder 재순위화 모델

한국어가 포함된 다국어 문서를 다룰 수 있는 `BAAI/bge-reranker-v2-m3`를 사용합니다. Colab GPU 메모리는 이미 생성 모델이 사용하므로 재순위화 모델은 **CPU**에 올립니다. 처음 실행할 때 약 2.3GB 모델 다운로드와 로딩 시간이 필요합니다.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

reranker_name = "BAAI/bge-reranker-v2-m3"
rr_tokenizer = AutoTokenizer.from_pretrained(reranker_name)
rr_model = AutoModelForSequenceClassification.from_pretrained(reranker_name).to("cpu")
rr_model.eval()

# 원래 질문을 Reranker의 기준으로 사용합니다.
# 세 검색 방식 모두 후보 8개를 가져와 상위 4개를 비교합니다.
candidate_retriever = vector_store.as_retriever(search_kwargs={"k": 8})
TOP_N = 2


In [ ]:
def rerank_candidates(question, candidate_docs, batch_size=4):
    """원래 질문과 각 후보를 함께 읽고 관련성 점수가 높은 순으로 정렬."""
    if not candidate_docs:
        return []

    pairs = [(question, doc.page_content) for doc in candidate_docs]
    scores = []

    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        inputs = rr_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        with torch.inference_mode():
            logits = rr_model(**inputs).logits.view(-1)
        scores.extend(logits.cpu().tolist())

    ranked = [
        {"original_rank": i + 1, "doc": doc, "score": float(score)}
        for i, (doc, score) in enumerate(zip(candidate_docs, scores))
    ]
    return sorted(ranked, key=lambda item: item["score"], reverse=True)


def answer_using_docs(question, docs):
    return answer_chain.invoke({
        "context": format_docs(docs),
        "question": question,
    })


def compare_reranking(question, search_query):
    """동일한 후보에서 재순위화 유무만 바꿔 두 답변을 생성."""
    candidate_docs = candidate_retriever.invoke(search_query)
    original_docs = candidate_docs[:TOP_N]
    ranked = rerank_candidates(question, candidate_docs)
    reranked_docs = [item["doc"] for item in ranked[:TOP_N]]

    return {
        "search_query": search_query,
        "candidate_docs": candidate_docs,
        "original_docs": original_docs,
        "ranked": ranked,
        "reranked_docs": reranked_docs,
        "answer_before": answer_using_docs(question, original_docs),
        "answer_after": answer_using_docs(question, reranked_docs),
    }


def pdf_page(doc):
    page = doc.metadata.get("page")
    return page + 1 if isinstance(page, int) else "알 수 없음"


def show_comparison(label, result):
    print(f"\n===== {label} =====")
    print("검색 입력:", result["search_query"])
    print("후보 8개 PDF 페이지:", [pdf_page(doc) for doc in result["candidate_docs"]])
    print("재순위화 전 Top 2:", [pdf_page(doc) for doc in result["original_docs"]])
    print("재순위화 후 Top 2:")
    for new_rank, item in enumerate(result["ranked"][:TOP_N], 1):
        print(
            f"  {new_rank}. PDF {pdf_page(item['doc'])}쪽 "
            f"(기존 {item['original_rank']}위, 점수 {item['score']:.3f})"
        )
        print("    ", item["doc"].page_content[:180].replace("\n", " "), "...")
    print("\n[재순위화 전 답변]\n", result["answer_before"])
    print("\n[재순위화 후 답변]\n", result["answer_after"])


## 5. 같은 질문으로 전후 비교

아래 질문을 설정한 뒤 세 실행 셀을 각각 돌리세요. Query Rewriting과 HyDE의 검색 입력은 각 셀에서 한 번만 생성합니다. 생성된 답변은 길 수 있으므로 필요한 방식부터 실행해도 됩니다.


In [ ]:
# question = "연말 정산 때 비거주자가 주의할 점을 알려 줘."
question = "2024년 개정 세법 중에 월세와 관련한 내용이 있을까?"
#question = "기부금 공제 때 주의할 점은?"


In [ ]:
baseline_result = compare_reranking(question, question)
show_comparison("기본 검색", baseline_result)


In [ ]:
rewrite_search_query = rewrite_query_chain.invoke({"question": question}).strip()
rewrite_result = compare_reranking(question, rewrite_search_query)
show_comparison("Query Rewriting", rewrite_result)


In [ ]:
hyde_search_query = hyde_query_chain.invoke({"question": question}).strip()
hyde_result = compare_reranking(question, hyde_search_query)
show_comparison("HyDE", hyde_result)


In [ ]:
# 세 실행 셀을 모두 돌린 후 요약표를 표시합니다.
from IPython.display import display
import pandas as pd

comparison = pd.DataFrame([
    {
        "검색 방식": label,
        "검색 입력": result["search_query"],
        "후보 페이지": [pdf_page(doc) for doc in result["candidate_docs"]],
        "전 Top 4": [pdf_page(doc) for doc in result["original_docs"]],
        "후 Top 4": [pdf_page(doc) for doc in result["reranked_docs"]],
        "전 답변": result["answer_before"],
        "후 답변": result["answer_after"],
    }
    for label, result in [
        ("기본 검색", baseline_result),
        ("Query Rewriting", rewrite_result),
        ("HyDE", hyde_result),
    ]
])
display(comparison)
